In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.dbutils import *

from datetime import datetime, date

dbutils.widgets.text('season',"2024","ipl season")
season = dbutils.widgets.get('season')

In [0]:
## read bronze data
if spark.catalog.tableExists("ipl_database.silver.extra_runs"):
    df_raw = spark.table("ipl_database.bronze.extra_runs_raw").filter(
        (to_date(col("updated_at")) >= date.today())
        &
        (col("runs").isNotNull()) 
        &
        (col("batting_team").isNotNull()) 
    ) \
    .withColumn(
        "season",
        lit(season)
    )
else:
    df_raw = spark.table("ipl_database.bronze.extra_runs_raw").withColumn(
        "season",
        lit(season)
    )
# df_raw.display()

## 00 Cleaning raw data

In [0]:
df = (
    df_raw
    .withColumn(
        "id",
        trim(col("id"))
    )
    .withColumn(
        "team",
        trim(col("batting_team"))
    )
    .withColumn(
        "extra_runs",
        col("runs").cast('int')
    )
    .withColumn(
        "season",
        col("season").cast('int')
    )
    .select("id", "season", "team", "extra_runs")
)

In [0]:
%skip
df.display()

## 01 Adding match UUID

In [0]:
match_uuid = spark.table("ipl_database.silver.match_details")

df_fin = (
  df.alias('a')
  .join(
    match_uuid.alias('b'),
    (col('a.id') == col('b.id')) & (col('a.season') == col('b.season')),
    'left'
  )
  .drop('id')
  .select('a.*','b.match_UUID')
)

In [0]:
%skip
df_fin.display()

## saving silver table

In [0]:
# if not spark.catalog.tableExists("ipl_database.silver.extra_runs"):
#   df_fin.write.format("delta").mode("overwrite").save(f"abfss://{container}@{storage_account}.dfs.core.windows.net/extra_runs")
#   spark.sql('''
#             create table ipl_database.silver.extra_runs
#             using delta
#             location 'abfss://silver@ipldatastorageaccount.dfs.core.windows.net/extra_runs'
#             ''')
#   print("write complete!!!")
# else:
#   df_fin.write.format("delta").mode("append").save(f"abfss://{container}@{storage_account}.dfs.core.windows.net/extra_runs")
#   spark.sql('refresh table ipl_database.silver.extra_runs')
#   print("Append Sucess!!!")

In [0]:
try:
    if spark.catalog.tableExists("ipl_database.silver.extra_runs"):
        df_fin.createOrReplaceTempView("df_fin")
        spark.sql(f'''
                merge into ipl_database.silver.extra_runs mc
                using df_fin nd
                on mc.team = nd.team and mc.season = nd.season and mc.match_UUID = nd.match_UUID
                -- when matched then update set *
                when not matched then insert *
                ''')
        # Upsert in not ideal here, switching to append process
        print("Table updated.")
    else:
        df_fin.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("ipl_database.silver.extra_runs")
        print("Table Created")
except Exception as e:
    print(f"Error while write/append,{e}")